In [ ]:
%pip install numpy pandas scipy scikit-learn

In [3]:
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


# Add imports needed by your chosen NLP techniques in your own cells.


Basicamente la celda de abajo es casi identica a la de la actividad 8.2, solamente se carga todo el dataset, antes se cargaba con load_parrtition por que eran puros .txt

In [7]:
activity_root = Path.cwd()
if activity_root.name == "notebooks":
    activity_root = activity_root.parent
dataset_root = activity_root / "data" / "SMSSpamCollection"


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(
           f"Expected 12,500 reviews in {directory}. "
                "Download and extract the complete IMDb dataset; see README.md."
        )
    df = pd.read_csv(path, sep="\t", header=None, names=["label", "text"], encoding="utf-8")
    df["label"] = (df["label"] == "spam").astype(int)
    return df


messages = load_dataset(dataset_root)

train_val, test_reviews = train_test_split(
    messages, test_size=0.15, random_state=42, stratify=messages["label"]
)
train_reviews, val_reviews = train_test_split(
    train_val, test_size=0.15 / 0.85, random_state=42, stratify=train_val["label"]
)
X_train = train_reviews["text"].copy()
X_val = val_reviews["text"].copy()
X_test = test_reviews["text"].copy()
y_train = train_reviews["label"].copy()
y_val = val_reviews["label"].copy()
y_test = test_reviews["label"].copy()
_fixed_inputs = tuple(series.copy(deep=True) for series in (X_train, X_val, X_test, y_train, y_val, y_test))
print("Training reviews:", len(train_reviews), "Validation reviews:", len(val_reviews), "Test reviews:", len(test_reviews))
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Validation labels:", y_val.value_counts().sort_index().to_dict())
print("Test labels:", y_test.value_counts().sort_index().to_dict())

Training reviews: 3900 Validation reviews: 836 Test reviews: 836
Training labels: {0: 3377, 1: 523}
Validation labels: {0: 724, 1: 112}
Test labels: {0: 724, 1: 112}


In [8]:
def _check_fixed_inputs():
    for actual, expected in zip((X_train, X_test, y_train, y_test), _fixed_inputs):
        pd.testing.assert_series_equal(actual, expected)


def _check_classifier(pipeline):
    assert isinstance(pipeline, Pipeline), "Create a sklearn Pipeline instance."
    assert "vectorizer" in pipeline.named_steps
    assert pipeline.steps[-1][0] == "classifier"
    classifier = pipeline.named_steps["classifier"]
    assert isinstance(classifier, LogisticRegression)
    assert classifier.get_params() == LogisticRegression(max_iter=1000).get_params()
    vectorizer = pipeline.named_steps["vectorizer"]
    assert isinstance(vectorizer, (CountVectorizer, TfidfVectorizer))
    return vectorizer, classifier


def _check_fitted_pipeline(pipeline, predictions, accuracy):
    _check_fixed_inputs()
    vectorizer, classifier = _check_classifier(pipeline)
    assert isinstance(predictions, np.ndarray)
    assert predictions.shape == (len(X_test),)
    assert set(predictions.tolist()).issubset({0, 1})
    np.testing.assert_array_equal(predictions, pipeline.predict(X_test))
    assert np.isscalar(accuracy) and 0.0 <= accuracy <= 1.0
    np.testing.assert_allclose(accuracy, accuracy_score(y_test, predictions), rtol=0, atol=1e-12)
    matrix = pipeline[:-1].transform(X_test)
    assert sparse.issparse(matrix), "Keep document-term matrices sparse."
    assert matrix.shape[0] == len(X_test)
    np.testing.assert_array_equal(classifier.classes_, [0, 1])

    # Teacher check: refit the declared design on training data only.
    reference = clone(pipeline)
    reference.fit(_fixed_inputs[0], _fixed_inputs[2])
    reference_vectorizer = reference.named_steps["vectorizer"]
    reference_classifier = reference.named_steps["classifier"]
    assert vectorizer.vocabulary_ == reference_vectorizer.vocabulary_
    if isinstance(vectorizer, TfidfVectorizer) and vectorizer.use_idf:
        np.testing.assert_allclose(vectorizer.idf_, reference_vectorizer.idf_)
    np.testing.assert_allclose(classifier.coef_, reference_classifier.coef_, rtol=1e-5, atol=1e-7)
    np.testing.assert_allclose(classifier.intercept_, reference_classifier.intercept_, rtol=1e-5, atol=1e-7)


def _check_independent(pipeline, others):
    for other in others:
        assert pipeline is not other, "Construct a new Pipeline instance."
        for step in ("vectorizer", "classifier"):
            assert pipeline.named_steps[step] is not other.named_steps[step], (
                f"Each pipeline needs its own new {step} instance."
            )
        assert (
            [(type(step), step.get_params(deep=False)) for _, step in pipeline.steps[:-1]]
            != [(type(step), step.get_params(deep=False)) for _, step in other.steps[:-1]]
        ), "Change preprocessing or vectorization, not just the variable name."


In [9]:
def normalize_text(text: str) -> str:
    text = text.lower();
    text = re.sub(r'[^a-z]', ' ', text)
    text = re.sub(r"\s+", " ", text).strip()    

    return text

In [10]:
baseline_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=normalize_text)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
_supplied_baseline = baseline_pipeline


In [16]:
baseline_pipeline.fit(X_train, y_train)
baseline_predictions = baseline_pipeline.predict(X_val)
baseline_accuracy = accuracy_score(y_val, baseline_predictions)

In [ ]:
alternative_1_pipeline = Pipeline([
    ("vectorizer",TfidfVectorizer(preprocessor=normalize_text)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
alternative_1_pipeline.fit(X_train, y_train)
alternative_1_predictions = alternative_1_pipeline.predict(X_test)
alternative_1_accuracy = accuracy_score(y_test, alternative_1_predictions)


puro codigo robado xd

In [13]:
alternative_2_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=normalize_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000)),
])
alternative_2_pipeline.fit(X_train, y_train)
alternative_2_predictions = alternative_2_pipeline.predict(X_test)
alternative_2_accuracy = accuracy_score(y_test, alternative_2_predictions)

In [ ]:
results = pd.DataFrame({
    "pipeline": ["baseline", "alternative_1", "alternative_2"],
    "accuracy": [baseline_accuracy, alternative_1_accuracy, alternative_2_accuracy],
})
display(results.style.format({"accuracy": "{:.2%}"}))

pipelines = {
    "baseline": baseline_pipeline,
    "alternative_1": alternative_1_pipeline,
    "alternative_2": alternative_2_pipeline,
}

names = ["baseline", "alternative_1", "alternative_2"]
pipelines = [baseline_pipeline, alternative_1_pipeline, alternative_2_pipeline]
accuracies = [baseline_accuracy, alternative_1_accuracy, alternative_2_accuracy]

results = pd.DataFrame({"pipeline": names, "accuracy": accuracies})
for name, acc in zip(names, accuracies):
    print(f"{name}: {acc:.2%}")

bestIndex = accuracies.index(max(accuracies))
best_name = names[bestIndex]
best_pipeline = pipelines[bestIndex]

print(f"Best pipeline: {best_name}")

,pipeline,accuracy
0,baseline,98.33%
1,alternative_1,96.29%
2,alternative_2,97.25%


baseline: 98.33%
alternative_1: 96.29%
alternative_2: 97.25%
Best pipeline: baseline
